Reproduce the pinned rank-stability manuscript in a dedicated Linux CPU runtime. The build fetches the same public commit into two separate shallow repositories and checks both source and PDF bytes. The returned archive contains both build logs and rendered pages. Record the connected time and actual compute-unit reading when available, retain and verify the archive, then disconnect the dedicated runtime.


In [ ]:
%%bash
set -euo pipefail
export TZ=UTC LC_ALL=C FORCE_SOURCE_DATE=1
source_path='papers/stationary-weighting354/research/2026-11-18-small-support/rank-defects-and-stability.tex'
source_sha256=aec662773a7c8e53bdda3039785204ea9bc2e771ad66bf5c19aae686b975feb1
root=$(mktemp -d /content/rank-stability.XXXXXXXX)
dest="$root/rank-stability-proof.zip"
export GIT_TERMINAL_PROMPT=0 DEBIAN_FRONTEND=noninteractive
repo=https://github.com/lluiseriksson/THE-ERIKSSON-PROGRAMME.git
pin=4184d9cf423f974101af1fa1059218c04025cc64
out="$root/proof"
mkdir "$out"
date -u +%FT%TZ > "$out/start-utc.txt"
if ! command -v pdflatex >/dev/null || ! kpsewhich lmodern.sty >/dev/null || ! kpsewhich microtype.sty >/dev/null || ! command -v pdftoppm >/dev/null || ! command -v zip >/dev/null; then
  apt-get -qq update > "$root/install.log" 2>&1
  apt-get -qq install -y --no-install-recommends texlive-latex-recommended lmodern poppler-utils zip >> "$root/install.log" 2>&1
fi
for n in 1 2; do
  build="$root/build$n"
  mkdir "$build"
  clone="$root/clone$n"
  git init -q "$clone"
  git -C "$clone" remote add origin "$repo"
  git -C "$clone" config remote.origin.promisor true
  git -C "$clone" config remote.origin.partialclonefilter blob:none
  git -C "$clone" -c protocol.version=2 fetch -q --depth=1 --filter=blob:none origin "$pin"
  actual=$(git -C "$clone" rev-parse FETCH_HEAD)
  test "$actual" = "$pin"
  git -C "$clone" show "$actual:$source_path" > "$build/manuscript.tex"
  SOURCE_DATE_EPOCH=$(git -C "$clone" show -s --format=%ct "$actual")
  export SOURCE_DATE_EPOCH
  printf '%s\n' "$actual" > "$out/build$n-source-commit.txt"
  (
    cd "$build"
    echo "$source_sha256  manuscript.tex" | sha256sum -c -
    printf '%s\n' '\pdfinfoomitdate=1\relax' '\pdftrailerid{}' '\pdfsuppressptexinfo=15\relax' '\input{manuscript.tex}' > driver.tex
    for pass in 1 2 3; do
      pdflatex -no-shell-escape -halt-on-error -interaction=nonstopmode driver.tex > "pass$pass.log" || { tail -40 "pass$pass.log"; exit 1; }
    done
    if grep -E 'There were undefined references|Citation .*undefined|Reference .*undefined|multiply defined|Rerun to get cross-references right' driver.log; then exit 2; fi
    cp driver.log "$out/build$n.log"
    sha256sum driver.pdf > "$out/build$n.sha256"
  )
done
cmp "$root/build1/driver.pdf" "$root/build2/driver.pdf"
cp "$root/build1/driver.pdf" "$out/manuscript.pdf"
pdfinfo "$out/manuscript.pdf" > "$out/pdfinfo.txt"
pdftotext -layout "$out/manuscript.pdf" "$out/text.txt"
mkdir "$out/pages"
pdftoppm -scale-to 1400 -jpeg -jpegopt quality=85 "$out/manuscript.pdf" "$out/pages/page"
printf '%s\n' '4184d9cf423f974101af1fa1059218c04025cc64' > "$out/source-commit.txt"
pdflatex --version > "$out/toolchain.txt"
date -u +%FT%TZ > "$out/end-utc.txt"
(cd "$out" && sha256sum manuscript.pdf build*.log build*.sha256 build*-source-commit.txt text.txt pdfinfo.txt source-commit.txt pages/*.jpg > SHA256SUMS)
test ! -e "$dest"
(cd "$out" && zip -q -r "$dest" .)
test "$(wc -c < "$dest")" -le 52428800
sha256sum "$dest" "$out/manuscript.pdf"
printf "COLAB_ARTIFACT=%s\n" "$dest"
